# T2.4 – DBRepo View Definitions

This notebook defines and creates DBRepo views for the Vienna Weather Wet-Month Prediction experiment.

The notebook creates a DBRepo-compatible view over the `weather_measurement` table. The full SQL view definitions, including the joined feature view and train/validation/test splits, are documented in `sql/create_views.sql`.

In [ ]:
import os
from getpass import getpass

from dbrepo.RestClient import RestClient
from dbrepo.api.dto import QueryDefinition

In [12]:
ENDPOINT = "https://test.dbrepo.tuwien.ac.at"

USERNAME = "azra1558"
PASSWORD = "Katalizator1558!"

DATABASE_ID = "899bfcba-7fec-40c9-9076-3a3a9372c844"

TABLE_IDS = {
    "weather_measurement": "2212bed4-ef8f-4d95-bb65-20b2adb28abd",
    "time_dimension": "9f4dc236-81cc-4bdb-93e7-a8b6ed5436f2",
    "station": "e6779029-ce40-4a9a-ad17-147e183dc757"
}

client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

print("Connected to DBRepo.")

Connected to DBRepo.


In [13]:
tables = client.get_tables(DATABASE_ID)

for table in tables:
    print(table.name, table.id)

weather_measurement 2212bed4-ef8f-4d95-bb65-20b2adb28abd
time_dimension 9f4dc236-81cc-4bdb-93e7-a8b6ed5436f2
station e6779029-ce40-4a9a-ad17-147e183dc757


In [14]:
def get_view_by_name(client, database_id, view_name):
    views = client.get_views(database_id)
    for view in views:
        if view.name == view_name:
            return view
    return None

In [ ]:
weather_feature_columns = [
    "weather_measurement.measurement_id",
    "weather_measurement.station_num",
    "weather_measurement.time_id",
    "weather_measurement.t_mean_c",
    "weather_measurement.t_max_c",
    "weather_measurement.t_min_c",
    "weather_measurement.mean_t_max_c",
    "weather_measurement.mean_t_min_c",
    "weather_measurement.p_mean_hpa",
    "weather_measurement.p_max_hpa",
    "weather_measurement.p_min_hpa",
    "weather_measurement.precp_sum_mm",
    "weather_measurement.num_precp_01",
    "weather_measurement.rel_hum_pct",
    "weather_measurement.rel_hum_max_pct",
    "weather_measurement.rel_hum_min_pct",
    "weather_measurement.wind_vel_ms",
    "weather_measurement.wind_vel_max_ms",
    "weather_measurement.num_wind_vel60",
    "weather_measurement.sun_h",
    "weather_measurement.num_clear",
    "weather_measurement.num_cloud",
    "weather_measurement.num_frost",
    "weather_measurement.num_ice",
    "weather_measurement.num_summer",
    "weather_measurement.num_heat",
]

view_name = "weather_measurement_features"

existing_view = get_view_by_name(client, DATABASE_ID, view_name)

if existing_view is not None:
    print("View already exists:", existing_view.name, existing_view.id)
else:
    query = QueryDefinition(
        datasources=["weather_measurement"],
        columns=weather_feature_columns
    )

    view = client.create_view(
        database_id=DATABASE_ID,
        name=view_name,
        query=query,
        is_public=False,
        is_schema_public=True
    )

    print("Created view:", view.name, view.id)

Created view: weather_measurement_features 063c704e-27c7-4432-a2d8-91a68747341a


In [19]:
views = client.get_views(DATABASE_ID)

print("DBRepo views:")
for view in views:
    row_count = client.get_view_data_count(DATABASE_ID, view.id)
    print(f"- {view.name}: {view.id} ({row_count} rows)")

DBRepo views:
- weather_measurement_features: 063c704e-27c7-4432-a2d8-91a68747341a (0 rows)


## Note on full SQL views

The DBRepo Python API was used to create the DBRepo-compatible view
`weather_measurement_features`.

The full SQL view definitions are documented in
`sql/create_views.sql`. These include the joined feature view, the train,
validation and test views, and a monthly precipitation summary view.

The views currently return zero rows because the actual data loading is handled
later in T2.5.